01. Pilot Dataset

02. Data Inspection

03. VIIRS Harmonization
Final QC

04. MODIS Preparation

05. Spatial Harmonization

06. Temporal Harmonization

07. Cross-Sensor Matching

08. Fire Event Generation

09. Validation

10. Analytics Features

11. Full Dataset Processing

12. Platform Integration

In [ ]:
modis_path='/content/drive/MyDrive/nasa-cairo-26/modis-Australia-2020.csv'
viirs_snpp_path='/content/drive/MyDrive/nasa-cairo-26/viirs-snpp-Australia-2020.csv'
viirs_jpss1_path='/content/drive/MyDrive/nasa-cairo-26/viirs-jpss1-Australia-2020.csv'

In [ ]:
import pandas as pd

In [ ]:
modis=pd.read_csv(modis_path)
viirs_snpp=pd.read_csv(viirs_snpp_path)
viirs_jpss1=pd.read_csv(viirs_jpss1_path)

In [ ]:
modis.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,-13.2051,143.1472,337.2,1.0,1.0,2020-01-01,50,Terra,MODIS,87,6.03,298.0,27.9,D,0
1,-14.4737,143.5811,351.4,1.0,1.0,2020-01-01,51,Terra,MODIS,95,6.03,300.4,55.8,D,0
2,-14.4828,143.5797,321.6,1.0,1.0,2020-01-01,51,Terra,MODIS,58,6.03,299.8,8.0,D,0
3,-14.7204,143.7693,343.0,1.0,1.0,2020-01-01,51,Terra,MODIS,89,6.03,300.4,35.5,D,0
4,-16.0143,145.1106,322.1,1.2,1.1,2020-01-01,51,Terra,MODIS,56,6.03,302.6,11.5,D,0


In [ ]:
viirs_snpp.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,-41.56622,147.95204,346.47,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,289.89,48.52,D,0
1,-41.56641,147.94772,352.94,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,295.35,48.52,D,0
2,-41.56716,147.93088,353.88,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,283.05,22.18,D,0
3,-41.56773,147.91777,340.81,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,285.54,6.93,D,0
4,-41.56793,147.91336,339.01,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,281.39,6.93,D,0


In [ ]:
viirs_jpss1.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,-41.66029,148.01045,334.00,0.38,0.36,2020-01-01,405,N20,VIIRS,n,2,278.68,4.48,D,0
1,-41.65956,148.01506,351.23,0.38,0.36,2020-01-01,405,N20,VIIRS,n,2,278.91,34.07,D,0
2,-41.66215,148.02060,356.20,0.38,0.36,2020-01-01,405,N20,VIIRS,l,2,275.79,34.07,D,0
3,-41.66360,148.01137,343.37,0.38,0.36,2020-01-01,405,N20,VIIRS,n,2,280.50,4.48,D,0
4,-41.66619,148.01691,367.00,0.38,0.36,2020-01-01,405,N20,VIIRS,l,2,331.93,352.66,D,0


# **Data Inspection**

In [ ]:

import pandas as pd

datasets = {
    "MODIS": modis,
    "VIIRS_SNPP": viirs_snpp,
    "VIIRS_JPSS1": viirs_jpss1
}

# Basic information
summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        ),
        "Exact_Duplicates": int(df.duplicated().sum())
    })

display(pd.DataFrame(summary))

# Compare schemas and data types
all_columns = sorted(
    set().union(*(df.columns for df in datasets.values()))
)

schema_report = pd.DataFrame({
    "Column": all_columns,
    **{
        name: [
            str(df[col].dtype) if col in df.columns else "MISSING"
            for col in all_columns
        ]
        for name, df in datasets.items()
    }
})

display(schema_report)

,Dataset,Rows,Columns,Memory_MB,Exact_Duplicates
0,MODIS,155524,15,45.15,0
1,VIIRS_SNPP,847691,15,277.29,0
2,VIIRS_JPSS1,829833,15,273.03,0


,Column,MODIS,VIIRS_SNPP,VIIRS_JPSS1
0,acq_date,object,object,object
1,acq_time,int64,int64,int64
2,bright_t31,float64,MISSING,MISSING
3,bright_ti4,MISSING,float64,float64
4,bright_ti5,MISSING,float64,float64
5,brightness,float64,MISSING,MISSING
6,confidence,int64,object,object
7,daynight,object,object,object
8,frp,float64,float64,float64
9,instrument,object,object,object


In [ ]:

target_year = 2020

for name, df in datasets.items():

    print(f"\n{'=' * 50}")
    print(f"DATASET: {name}")
    print("=" * 50)

    # Missing values
    missing = df.isna().sum()
    print("\nMissing Values:")
    print(missing[missing > 0].to_string()
          if (missing > 0).any() else "None")

    # Dates
    dates = pd.to_datetime(
        df["acq_date"], errors="coerce", utc=True
    )

    print("\nDate Range:")
    print("Start:", dates.min())
    print("End:", dates.max())
    print("Invalid dates:", dates.isna().sum())
    print("Outside target year:",
          (dates.notna() & dates.dt.year.ne(target_year)).sum())

    # Time validation - HHMM
    times = (
        df["acq_time"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(4)
    )

    hours = pd.to_numeric(times.str[:2], errors="coerce")
    minutes = pd.to_numeric(times.str[2:], errors="coerce")

    valid_time = (
        times.str.fullmatch(r"\d{4}").fillna(False)
        & hours.between(0, 23)
        & minutes.between(0, 59)
    )

    print("Invalid times:", (~valid_time).sum())

    # Coordinates
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")

    invalid_coords = (
        ~lat.between(-90, 90) |
        ~lon.between(-180, 180)
    )

    print("\nInvalid Coordinates:", invalid_coords.sum())
    print("Latitude Range:", lat.min(), "to", lat.max())
    print("Longitude Range:", lon.min(), "to", lon.max())

    # FRP
    frp = pd.to_numeric(df["frp"], errors="coerce")
    print("\nInvalid/Missing FRP:", frp.isna().sum())
    print("Negative FRP:", frp.lt(0).sum())

    # Categorical metadata
    for col in [
        "satellite", "instrument", "version",
        "confidence", "daynight", "type"
    ]:
        if col in df.columns:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).to_string())



DATASET: MODIS

Missing Values:
None

Date Range:
Start: 2020-01-01 00:00:00+00:00
End: 2020-12-31 00:00:00+00:00
Invalid dates: 0
Outside target year: 0
Invalid times: 0

Invalid Coordinates: 0
Latitude Range: -43.5006 to -9.2741
Longitude Range: 113.7342 to 153.5515

Invalid/Missing FRP: 0
Negative FRP: 3

satellite:
satellite
Aqua     93262
Terra    62262

instrument:
instrument
MODIS    155524

version:
version
6.03    155524

confidence:
confidence
100    19509
75      2667
74      2586
77      2525
70      2522
76      2518
81      2493
71      2472
79      2471
73      2451
72      2444
69      2438
68      2436
82      2432
80      2429
66      2417
78      2404
84      2320
67      2308
63      2298
65      2280
87      2279
85      2273
83      2270
62      2247
64      2208
61      2148
86      2076
60      2071
58      2031
59      2023
94      2015
88      1986
57      1933
89      1930
90      1918
91      1850
56      1789
55      1786
92      1757
54      1742
53      

Problem : we find `Negative FRP`

In [ ]:
for name, df in datasets.items():

    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)

    negative = df[df["frp"] < 0]

    print("Negative FRP:", len(negative))
    print("Zero FRP:", (df["frp"] == 0).sum())
    print("Minimum FRP:", df["frp"].min())

    print("\nNegative FRP Values:")
    print(negative["frp"].value_counts())

    print("\nNegative FRP by Type:")
    print(negative["type"].value_counts())

    print("\nNegative FRP by Day/Night:")
    print(negative["daynight"].value_counts())

    print("\nSample Records:")
    display(negative.head(5))


MODIS
Negative FRP: 3
Zero FRP: 1157
Minimum FRP: -29.9

Negative FRP Values:
frp
-22.3    1
-9.3     1
-29.9    1
Name: count, dtype: int64

Negative FRP by Type:
type
0    3
Name: count, dtype: int64

Negative FRP by Day/Night:
daynight
D    3
Name: count, dtype: int64

Sample Records:


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
33784,-32.6436,123.3889,373.3,1.0,1.0,2020-01-29,538,Aqua,MODIS,0,6.03,331.2,-22.3,D,0
33788,-32.6344,123.3872,371.5,1.0,1.0,2020-01-29,538,Aqua,MODIS,0,6.03,335.6,-9.3,D,0
33795,-32.6147,123.3947,360.7,1.0,1.0,2020-01-29,538,Aqua,MODIS,0,6.03,336.7,-29.9,D,0



VIIRS_SNPP
Negative FRP: 155
Zero FRP: 351
Minimum FRP: -3.66

Negative FRP Values:
frp
-0.04    4
-0.34    4
-0.30    4
-0.65    3
-0.08    3
        ..
-0.98    1
-2.06    1
-1.95    1
-1.38    1
-1.86    1
Name: count, Length: 113, dtype: int64

Negative FRP by Type:
type
0    155
Name: count, dtype: int64

Negative FRP by Day/Night:
daynight
D    155
Name: count, dtype: int64

Sample Records:


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
63358,-37.14693,148.38875,367.0,0.39,0.36,2020-01-04,400,N,VIIRS,l,2,378.07,-0.08,D,0
63596,-37.10325,148.31465,367.0,0.39,0.36,2020-01-04,400,N,VIIRS,l,2,351.18,-0.24,D,0
63597,-37.10395,148.31024,367.0,0.39,0.36,2020-01-04,400,N,VIIRS,l,2,365.31,-0.04,D,0
65336,-37.10741,147.09706,367.0,0.39,0.36,2020-01-04,400,N,VIIRS,l,2,370.09,-0.39,D,0
65495,-37.05201,147.37663,367.0,0.38,0.36,2020-01-04,400,N,VIIRS,l,2,372.22,-0.94,D,0



VIIRS_JPSS1
Negative FRP: 8
Zero FRP: 248
Minimum FRP: -2.96

Negative FRP Values:
frp
-2.77    1
-1.01    1
-0.13    1
-0.41    1
-0.15    1
-1.37    1
-2.96    1
-2.15    1
Name: count, dtype: int64

Negative FRP by Type:
type
0    8
Name: count, dtype: int64

Negative FRP by Day/Night:
daynight
D    8
Name: count, dtype: int64

Sample Records:


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
56076,-36.01140,148.31886,367.0,0.39,0.59,2020-01-04,310,N20,VIIRS,l,2,330.69,-2.77,D,0
56092,-36.00394,148.35349,367.0,0.39,0.59,2020-01-04,310,N20,VIIRS,l,2,367.17,-1.01,D,0
62318,-36.45173,149.65555,367.0,0.33,0.55,2020-01-04,310,N20,VIIRS,l,2,367.30,-0.13,D,0
71004,-36.44251,147.77501,367.0,0.43,0.62,2020-01-04,451,N20,VIIRS,l,2,347.04,-0.41,D,0
72290,-35.95522,148.41219,367.0,0.48,0.65,2020-01-04,451,N20,VIIRS,l,2,342.79,-0.15,D,0


# **VIIRS Data Integration & Preprocessing**

In [ ]:
import pandas as pd
import numpy as np

# Confirm identical VIIRS schemas
assert set(viirs_snpp.columns) == set(viirs_jpss1.columns)

def prepare_viirs(df, source, satellite_name, prefix):
    result = df.copy()

    # Preserve provenance
    result["source_dataset"] = source
    result["satellite_name"] = satellite_name
    result["source_row"] = np.arange(len(result))

    # Unique detection ID for this pilot
    result["detection_id"] = (
        prefix + "_" +
        (result["source_row"] + 1).astype(str)
    )

    return result

snpp_ready = prepare_viirs(
    viirs_snpp,
    "VIIRS_SNPP",
    "Suomi NPP",
    "AU2020_SNPP"
)

jpss1_ready = prepare_viirs(
    viirs_jpss1,
    "VIIRS_JPSS1",
    "NOAA-20",
    "AU2020_JPSS1"
)

# Combine records without removing any detections
viirs_unified = pd.concat(
    [snpp_ready, jpss1_ready],
    ignore_index=True
)

# Verify record preservation
expected_rows = len(viirs_snpp) + len(viirs_jpss1)

assert len(viirs_unified) == expected_rows
assert viirs_unified["detection_id"].is_unique

print("Unified VIIRS Records:", len(viirs_unified))
print("\nRecords by Satellite:")
print(viirs_unified["source_dataset"].value_counts())

display(viirs_unified.head())

Unified VIIRS Records: 1677524

Records by Satellite:
source_dataset
VIIRS_SNPP     847691
VIIRS_JPSS1    829833
Name: count, dtype: int64


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,source_dataset,satellite_name,source_row,detection_id
0,-41.56622,147.95204,346.47,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,289.89,48.52,D,0,VIIRS_SNPP,Suomi NPP,0,AU2020_SNPP_1
1,-41.56641,147.94772,352.94,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,295.35,48.52,D,0,VIIRS_SNPP,Suomi NPP,1,AU2020_SNPP_2
2,-41.56716,147.93088,353.88,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,283.05,22.18,D,0,VIIRS_SNPP,Suomi NPP,2,AU2020_SNPP_3
3,-41.56773,147.91777,340.81,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,285.54,6.93,D,0,VIIRS_SNPP,Suomi NPP,3,AU2020_SNPP_4
4,-41.56793,147.91336,339.01,0.37,0.58,2020-01-01,315,N,VIIRS,n,2,281.39,6.93,D,0,VIIRS_SNPP,Suomi NPP,4,AU2020_SNPP_5


In [ ]:
# 1. Create UTC datetime
times = pd.to_numeric(
    viirs_unified["acq_time"], errors="coerce"
)

hours = times // 100
minutes = times % 100

valid_time = (
    times.notna()
    & hours.between(0, 23)
    & minutes.between(0, 59)
)

assert valid_time.all(), "Invalid acquisition times found"

dates = pd.to_datetime(
    viirs_unified["acq_date"],
    format="%Y-%m-%d",
    errors="coerce",
    utc=True
)

assert dates.notna().all(), "Invalid dates found"

viirs_unified["acquisition_datetime_utc"] = (
    dates
    + pd.to_timedelta(hours, unit="h")
    + pd.to_timedelta(minutes, unit="m")
)

# 2. Classify FRP values
frp = pd.to_numeric(
    viirs_unified["frp"], errors="coerce"
)

viirs_unified["frp_status"] = np.select(
    [
        frp.isna(),
        frp < 0,
        frp == 0
    ],
    [
        "missing",
        "negative",
        "zero"
    ],
    default="positive"
)

# Temporarily use positive FRP only for quantitative analysis
viirs_unified["frp_mw_for_analysis"] = frp.where(frp > 0)

# 3. Validation
assert len(viirs_unified) == 1677524
assert viirs_unified["detection_id"].is_unique

print("Datetime Range:")
print(viirs_unified["acquisition_datetime_utc"].min())
print(viirs_unified["acquisition_datetime_utc"].max())

print("\nFRP Status by Satellite:")
display(
    pd.crosstab(
        viirs_unified["source_dataset"],
        viirs_unified["frp_status"]
    )
)

print("\nSample:")
display(
    viirs_unified[
        [
            "detection_id",
            "satellite_name",
            "acquisition_datetime_utc",
            "frp",
            "frp_status",
            "frp_mw_for_analysis"
        ]
    ].head()
)


Datetime Range:
2020-01-01 03:15:00+00:00
2020-12-31 18:05:00+00:00

FRP Status by Satellite:


frp_status,negative,positive,zero
source_dataset,,,
VIIRS_JPSS1,8,829577,248
VIIRS_SNPP,155,847185,351



Sample:


,detection_id,satellite_name,acquisition_datetime_utc,frp,frp_status,frp_mw_for_analysis
0,AU2020_SNPP_1,Suomi NPP,2020-01-01 03:15:00+00:00,48.52,positive,48.52
1,AU2020_SNPP_2,Suomi NPP,2020-01-01 03:15:00+00:00,48.52,positive,48.52
2,AU2020_SNPP_3,Suomi NPP,2020-01-01 03:15:00+00:00,22.18,positive,22.18
3,AU2020_SNPP_4,Suomi NPP,2020-01-01 03:15:00+00:00,6.93,positive,6.93
4,AU2020_SNPP_5,Suomi NPP,2020-01-01 03:15:00+00:00,6.93,positive,6.93


In [ ]:
df = viirs_unified

expected_count = len(viirs_snpp) + len(viirs_jpss1)

expected_satellite = df["source_dataset"].map({
    "VIIRS_SNPP": "N",
    "VIIRS_JPSS1": "N20"
})

dates = df["acquisition_datetime_utc"]

checks = {
    "Record count preserved":
        len(df) == expected_count,

    "Unique detection IDs":
        df["detection_id"].is_unique,

    "Source rows unique":
        not df.duplicated(
            subset=["source_dataset", "source_row"]
        ).any(),

    "Correct satellite mapping":
        df["satellite"].eq(expected_satellite).all(),

    "Valid UTC datetimes":
        dates.notna().all()
        and str(dates.dt.tz) == "UTC",

    "Correct year":
        dates.dt.year.eq(2020).all(),

    "Valid coordinates":
        df["latitude"].between(-90, 90).all()
        and df["longitude"].between(-180, 180).all(),

    "Valid confidence categories":
        df["confidence"].isin(["l", "n", "h"]).all(),

    "Valid FRP status":
        (
            (df["frp"] < 0).eq(
                df["frp_status"].eq("negative")
            ).all()
            and
            (df["frp"] == 0).eq(
                df["frp_status"].eq("zero")
            ).all()
            and
            (df["frp"] > 0).eq(
                df["frp_status"].eq("positive")
            ).all()
        ),

    "Non-positive FRP excluded from analysis":
        df.loc[
            df["frp"] <= 0,
            "frp_mw_for_analysis"
        ].isna().all()
}

quality_report = pd.DataFrame([
    {
        "Check": name,
        "Status": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

display(quality_report)

if not all(checks.values()):
    raise ValueError("Quality checks failed!")

print("All quality checks passed.")

,Check,Status
0,Record count preserved,PASS
1,Unique detection IDs,PASS
2,Source rows unique,PASS
3,Correct satellite mapping,PASS
4,Valid UTC datetimes,PASS
5,Correct year,PASS
6,Valid coordinates,PASS
7,Valid confidence categories,PASS
8,Valid FRP status,PASS
9,Non-positive FRP excluded from analysis,PASS


All quality checks passed.


In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

output_dir = Path("/content/drive/MyDrive/nasa-cairo-26/Clean Data")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "VIIRS_UNIFIED_Australia_2020.csv"
parquet_path = output_dir / "VIIRS_UNIFIED_Australia_2020.parquet"

# Save CSV
viirs_unified.to_csv(
    csv_path,
    index=False
)

# Save compressed Parquet
viirs_unified.to_parquet(
    parquet_path,
    index=False,
    compression="snappy"
)

# Verify saved Parquet row count
saved_rows = pq.ParquetFile(parquet_path).metadata.num_rows
assert saved_rows == len(viirs_unified)

print("Files saved successfully!")
print("CSV:", csv_path)
print("Parquet:", parquet_path)
print("Records:", saved_rows)

Files saved successfully!
CSV: /content/drive/MyDrive/nasa-cairo-26/Clean Data/VIIRS_UNIFIED_Australia_2020.csv
Parquet: /content/drive/MyDrive/nasa-cairo-26/Clean Data/VIIRS_UNIFIED_Australia_2020.parquet
Records: 1677524


# **MODIS Preparation**

In [ ]:
import pandas as pd
import numpy as np

# Work on a copy to protect original data
modis_ready = modis.copy()

# 1. Source metadata
modis_ready["source_dataset"] = "MODIS"
modis_ready["satellite_name"] = modis_ready["satellite"]
modis_ready["source_row"] = np.arange(len(modis_ready))

# 2. Unique Detection ID
modis_ready["detection_id"] = (
    "AU2020_MODIS_" +
    (modis_ready["source_row"] + 1).astype(str)
)

# 3. UTC Datetime
times = pd.to_numeric(
    modis_ready["acq_time"], errors="coerce"
)

hours = times // 100
minutes = times % 100

valid_time = (
    times.notna()
    & (times % 1 == 0)
    & hours.between(0, 23)
    & minutes.between(0, 59)
)

assert valid_time.all(), "Invalid acquisition times"

dates = pd.to_datetime(
    modis_ready["acq_date"],
    format="%Y-%m-%d",
    errors="coerce",
    utc=True
)

assert dates.notna().all(), "Invalid acquisition dates"

modis_ready["acquisition_datetime_utc"] = (
    dates
    + pd.to_timedelta(hours, unit="h")
    + pd.to_timedelta(minutes, unit="m")
)

# 4. Preserve native confidence
modis_ready["confidence_native"] = (
    modis_ready["confidence"].astype("string")
)

# 5. FRP quality classification
frp = pd.to_numeric(
    modis_ready["frp"], errors="coerce"
)

modis_ready["frp_status"] = np.select(
    [
        frp.isna(),
        frp < 0,
        frp == 0
    ],
    [
        "missing",
        "negative",
        "zero"
    ],
    default="positive"
)

# Positive values for quantitative analysis
modis_ready["frp_mw_for_analysis"] = frp.where(frp > 0)

# 6. Initial checks
print("Total Records:", len(modis_ready))

print("\nRecords by Satellite:")
print(modis_ready["satellite_name"].value_counts())

print("\nFRP Status:")
print(modis_ready["frp_status"].value_counts())

print("\nDatetime Range:")
print(modis_ready["acquisition_datetime_utc"].min())
print(modis_ready["acquisition_datetime_utc"].max())

display(modis_ready.head())

Total Records: 155524

Records by Satellite:
satellite_name
Aqua     93262
Terra    62262
Name: count, dtype: int64

FRP Status:
frp_status
positive    154364
zero          1157
negative         3
Name: count, dtype: int64

Datetime Range:
2020-01-01 00:50:00+00:00
2020-12-31 17:21:00+00:00


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,daynight,type,source_dataset,satellite_name,source_row,detection_id,acquisition_datetime_utc,confidence_native,frp_status,frp_mw_for_analysis
0,-13.2051,143.1472,337.2,1.0,1.0,2020-01-01,50,Terra,MODIS,87,...,D,0,MODIS,Terra,0,AU2020_MODIS_1,2020-01-01 00:50:00+00:00,87,positive,27.9
1,-14.4737,143.5811,351.4,1.0,1.0,2020-01-01,51,Terra,MODIS,95,...,D,0,MODIS,Terra,1,AU2020_MODIS_2,2020-01-01 00:51:00+00:00,95,positive,55.8
2,-14.4828,143.5797,321.6,1.0,1.0,2020-01-01,51,Terra,MODIS,58,...,D,0,MODIS,Terra,2,AU2020_MODIS_3,2020-01-01 00:51:00+00:00,58,positive,8.0
3,-14.7204,143.7693,343.0,1.0,1.0,2020-01-01,51,Terra,MODIS,89,...,D,0,MODIS,Terra,3,AU2020_MODIS_4,2020-01-01 00:51:00+00:00,89,positive,35.5
4,-16.0143,145.1106,322.1,1.2,1.1,2020-01-01,51,Terra,MODIS,56,...,D,0,MODIS,Terra,4,AU2020_MODIS_5,2020-01-01 00:51:00+00:00,56,positive,11.5


In [ ]:
df = modis_ready

checks = {
    "Record count preserved":
        len(df) == len(modis),

    "Unique Detection IDs":
        df["detection_id"].is_unique,

    "Unique source rows":
        df["source_row"].is_unique,

    "Valid UTC timestamps":
        df["acquisition_datetime_utc"].notna().all()
        and str(df["acquisition_datetime_utc"].dt.tz) == "UTC",

    "Correct year":
        df["acquisition_datetime_utc"].dt.year.eq(2020).all(),

    "Valid coordinates":
        df["latitude"].between(-90, 90).all()
        and df["longitude"].between(-180, 180).all(),

    "Valid MODIS satellites":
        df["satellite"].isin(["Terra", "Aqua"]).all(),

    "Valid confidence range":
        df["confidence"].between(0, 100).all(),

    "Correct FRP status":
        (
            (df["frp"] < 0).eq(
                df["frp_status"].eq("negative")
            ).all()
            and
            (df["frp"] == 0).eq(
                df["frp_status"].eq("zero")
            ).all()
            and
            (df["frp"] > 0).eq(
                df["frp_status"].eq("positive")
            ).all()
        ),

    "Non-positive FRP excluded from analysis":
        df.loc[
            df["frp"] <= 0,
            "frp_mw_for_analysis"
        ].isna().all(),

    "Expected FRP counts":
        df["frp_status"].value_counts().to_dict() == {
            "positive": 154364,
            "zero": 1157,
            "negative": 3
        }
}

quality_report = pd.DataFrame([
    {"Check": name, "Status": "PASS" if ok else "FAIL"}
    for name, ok in checks.items()
])

display(quality_report)

if not all(checks.values()):
    raise ValueError("MODIS quality checks failed")

print("All MODIS quality checks passed!")

,Check,Status
0,Record count preserved,PASS
1,Unique Detection IDs,PASS
2,Unique source rows,PASS
3,Valid UTC timestamps,PASS
4,Correct year,PASS
5,Valid coordinates,PASS
6,Valid MODIS satellites,PASS
7,Valid confidence range,PASS
8,Correct FRP status,PASS
9,Non-positive FRP excluded from analysis,PASS


All MODIS quality checks passed!


In [ ]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/nasa-cairo-26/Clean Data")
output_dir.mkdir(parents=True, exist_ok=True)

assert all(checks.values()), "Quality checks must pass first"

modis_ready.to_csv(
    output_dir / "MODIS_PREPARED_Australia_2020.csv",
    index=False
)

modis_ready.to_parquet(
    output_dir / "MODIS_PREPARED_Australia_2020.parquet",
    index=False,
    compression="snappy"
)

# Verify exported Parquet
saved = pd.read_parquet(
    output_dir / "MODIS_PREPARED_Australia_2020.parquet"
)

assert len(saved) == len(modis_ready)
assert saved["detection_id"].is_unique

print("MODIS saved successfully!")
print("Records:", len(saved))

MODIS saved successfully!
Records: 155524


# **MODIS & VIIRS Harmonization**

## Spatial Harmonization

## Temporal Harmonization

## Cross-Sensor Matching

## Fire Event Generation

## Validation